# Part 3: Pipeline Execution

Now that the database is configured, we'll run the deterioration pipeline to extract features and outcomes.

---

## What Happens During Pipeline Execution?

The pipeline creates these temporary tables in your PostgreSQL database:

1. **tmp_base_ed_cohort** - Base cohort of ED visits (adults 18+)
2. **tmp_ed_event_log** - Clinical events (ICU, death, interventions)
3. **tmp_ed_outcomes** - Binary outcome labels with time windows
4. **tmp_features_w1** - Features from first 1 hour
5. **tmp_features_w6** - Features from first 6 hours
6. **tmp_features_w24** - Features from first 24 hours
7. **tmp_ecg_features_w1** - ECG features (1-hour window)
8. **tmp_ecg_features_w6** - ECG features (6-hour window)

**Estimated time:** 10-30 minutes depending on data size

---

## Step 3.1: Load Configuration and Connect

In [2]:
#cd c:\Users\Lab\Desktop\Cardiac_Deterioration_Pipeline

c:\Users\Lab\Desktop\Cardiac_Deterioration_Pipeline


UsageError: Environment does not have key: setup_venv\Scripts\activate


In [5]:
# Imports
import os
os.environ['PGPASSWORD'] = "---"  # Replace with your actual password
import sys
import time
from datetime import datetime
from src.utils import load_yaml, setup_logging
from src.db import get_conn, check_connection

# Setup logging
setup_logging(verbose=True)

# Load config
cfg = load_yaml("config/config.yaml")

# Verify password is set
if 'PGPASSWORD' not in os.environ:
    import getpass
    os.environ['PGPASSWORD'] = getpass.getpass("Enter PostgreSQL password: ")

# Test connection
print("Testing database connection...")
if check_connection(cfg):
    print("✓ Connection OK\n")
else:
    print("✗ Connection failed. Check your config and password.")
    sys.exit(1)

2026-02-05 17:23:06,092 - src.utils - INFO - Logging initialized (level: DEBUG)
2026-02-05 17:23:06,094 - src.utils - INFO - Loaded configuration from: config/config.yaml
2026-02-05 17:23:06,131 - src.db - INFO - Connected to database: mimiciv @ localhost
2026-02-05 17:23:06,132 - src.db - INFO - PostgreSQL version: PostgreSQL 18.1 on x86_64-windows, compiled by msvc-19.44.35221, 64-bit


Testing database connection...
✓ Connection OK



## Step 3.2: Run Full Pipeline

This will execute all pipeline steps sequentially:

⚠️ **Warning:** This may take some time. Monitor progress below.

powershell command to check live progress:

& "C:\Program Files\PostgreSQL\18\bin\psql.exe" -U postgres -d mimiciv -c "SELECT pid, state, query_start, now() - query_start AS running_for, query FROM pg_stat_activity WHERE state = 'active' ORDER BY query_start DESC;"

ventilation event takes around ~30 mins

Running using powershell commands maybe quicker

### Diagnostic: Check ACS and Revascularization Events

If you're getting 0 events for ACS or Revascularization, run this diagnostic before running the full pipeline:


In [7]:
import pandas as pd

# Test ACS extraction
print("=" * 70)
print("DIAGNOSTIC: ACS Event Extraction")
print("=" * 70)

conn = get_conn(cfg)

# Check if ACS diagnoses exist in the hospital module
acs_check_sql = """
SELECT 
    COUNT(*) as acs_diagnosis_count,
    COUNT(DISTINCT hadm_id) as admissions_with_acs
FROM {hosp_schema}.diagnoses_icd d
INNER JOIN {hosp_schema}.d_icd_diagnoses dicd
  ON d.icd_code = dicd.icd_code
  AND d.icd_version = dicd.icd_version
WHERE dicd.icd_version = 10
  AND (
    dicd.icd_code LIKE 'I21%'   -- STEMI
    OR dicd.icd_code LIKE 'I22%'  -- Subsequent STEMI
    OR dicd.icd_code LIKE 'I24.0' -- Acute coronary thrombosis
    OR dicd.icd_code = 'I20.0'    -- Unstable angina
  )
""".format(hosp_schema=cfg['schemas']['hosp'])

print("\nChecking for ACS diagnoses in hospital module...")
try:
    df_acs = pd.read_sql(acs_check_sql, conn)
    print(df_acs.to_string(index=False))
except Exception as e:
    print(f"✗ Error: {e}")

# Check if those admissions are in the base cohort with hadm_id
print("\nChecking if ACS admissions are in base cohort...")
base_cohort_acs_sql = f"""
SELECT 
    COUNT(*) as base_cohort_count,
    COUNT(DISTINCT hadm_id) as admissions_with_hadm_id
FROM tmp_base_ed_cohort b
WHERE b.hadm_id IS NOT NULL
"""

try:
    df_base = pd.read_sql(base_cohort_acs_sql, conn)
    print(df_base.to_string(index=False))
except Exception as e:
    print(f"✗ Error: {e}")

conn.close()

print("\n" + "=" * 70)
print("DIAGNOSTIC: Revascularization Event Extraction")
print("=" * 70)

conn = get_conn(cfg)

# Check if revascularization procedures exist
revasc_check_sql = """
SELECT 
    COUNT(*) as procedure_count,
    COUNT(DISTINCT hadm_id) as admissions_with_procedures
FROM {hosp_schema}.procedures_icd p
INNER JOIN {hosp_schema}.d_icd_procedures proc
  ON p.icd_code = proc.icd_code
  AND p.icd_version = proc.icd_version
WHERE proc.icd_version = 10
  AND (
    proc.icd_code LIKE '0270%'   -- Dilation (PCI)
    OR proc.icd_code LIKE '021%'   -- Bypass (CABG)
  )
""".format(hosp_schema=cfg['schemas']['hosp'])

print("\nChecking for revascularization procedures...")
try:
    df_revasc = pd.read_sql(revasc_check_sql, conn)
    print(df_revasc.to_string(index=False))
except Exception as e:
    print(f"✗ Error: {e}")

conn.close()


2026-02-05 17:40:17,879 - src.db - INFO - Connected to database: mimiciv @ localhost


DIAGNOSTIC: ACS Event Extraction

Checking for ACS diagnoses in hospital module...


C:\Users\Lab\AppData\Local\Temp\ipykernel_14212\4132106723.py:30: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_acs = pd.read_sql(acs_check_sql, conn)
C:\Users\Lab\AppData\Local\Temp\ipykernel_14212\4132106723.py:46: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_base = pd.read_sql(base_cohort_acs_sql, conn)
2026-02-05 17:40:19,069 - src.db - INFO - Connected to database: mimiciv @ localhost


 acs_diagnosis_count  admissions_with_acs
               10035                 9840

Checking if ACS admissions are in base cohort...
 base_cohort_count  admissions_with_hadm_id
            202990                   202415

DIAGNOSTIC: Revascularization Event Extraction

Checking for revascularization procedures...


C:\Users\Lab\AppData\Local\Temp\ipykernel_14212\4132106723.py:77: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_revasc = pd.read_sql(revasc_check_sql, conn)


 procedure_count  admissions_with_procedures
           10257                        6115


In [8]:

# Test ACS SQL directly with template substitution
print("\n" + "=" * 70)
print("TEST: Running ACS Event SQL Directly")
print("=" * 70)

from src.utils import read_sql, render_sql_template, get_sql_mapping

conn = get_conn(cfg)

# Load and render the ACS SQL
acs_sql = read_sql("sql/14_event_acs.sql")
mapping = get_sql_mapping(cfg)
acs_sql_rendered = render_sql_template(acs_sql, mapping)

print("\nRendered ACS SQL (first 500 chars):")
print(acs_sql_rendered[:500])
print("\n...")

# Test the rendered SQL
print("\nExecuting ACS SQL...")
try:
    import pandas as pd
    df_test = pd.read_sql(acs_sql_rendered, conn)
    print(f"✓ SQL executed successfully: {len(df_test)} rows returned")
    if len(df_test) > 0:
        print("\nFirst few rows:")
        print(df_test.head())
    else:
        print("⚠️  0 rows returned - checking why...")
except Exception as e:
    print(f"✗ SQL execution failed: {e}")

conn.close()


2026-02-05 17:41:10,354 - src.db - INFO - Connected to database: mimiciv @ localhost
2026-02-05 17:41:10,356 - src.utils - DEBUG - Read SQL from: sql/14_event_acs.sql



TEST: Running ACS Event SQL Directly

Rendered ACS SQL (first 500 chars):
-- Event: Acute Coronary Syndrome (ACS)
-- Identifies ACS diagnoses from ICD codes
-- Includes STEMI, NSTEMI, unstable angina
-- NOTE: Timing is limited - conservatively anchored to ED intime

WITH acs_icd_codes AS (
  -- ICD-10 codes for ACS
  SELECT icd_code, icd_version, long_title
  FROM mimiciv_hosp.d_icd_diagnoses
  WHERE icd_version = 10
    AND (
      icd_code LIKE 'I21%'     -- ST elevation myocardial infarction
      OR icd_code LIKE 'I22%'  -- Subsequent STEMI
      OR icd_code LIKE 

...

Executing ACS SQL...


C:\Users\Lab\AppData\Local\Temp\ipykernel_14212\4256890928.py:23: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_test = pd.read_sql(acs_sql_rendered, conn)


✓ SQL executed successfully: 6195 rows returned

First few rows:
   subject_id   stay_id   hadm_id event_type          event_time  \
0    10000764  35420907  27897940        ACS 2132-10-14 19:31:00   
1    10000980  30905710  26913865        ACS 2189-06-27 06:25:00   
2    10002013  30061882  24760295        ACS 2160-07-10 15:17:00   
3    10003502  30066775  20459702        ACS 2166-02-15 09:09:00   
4    10007058  38967799  22954658        ACS 2167-11-07 17:57:00   

                          source_table  \
0  hosp.diagnoses_icd (timing limited)   
1  hosp.diagnoses_icd (timing limited)   
2  hosp.diagnoses_icd (timing limited)   
3  hosp.diagnoses_icd (timing limited)   
4  hosp.diagnoses_icd (timing limited)   

                                        event_detail  
0  Subendocardial infarction, initial episode of ...  
1  Subendocardial infarction, initial episode of ...  
2  Subendocardial infarction, initial episode of ...  
3  Subendocardial infarction, initial episode of ... 

In [9]:

# Check what's actually in the event log table for ACS/Revascularization
print("\n" + "=" * 70)
print("DIAGNOSTIC: Check Event Log Contents")
print("=" * 70)

conn = get_conn(cfg)

# Check all event types in the event log
event_types_sql = """
SELECT 
    event_type,
    COUNT(*) as event_count
FROM tmp_ed_event_log
GROUP BY event_type
ORDER BY event_count DESC
"""

print("\nEvent types in event log:")
try:
    df_events = pd.read_sql(event_types_sql, conn)
    print(df_events.to_string(index=False))
except Exception as e:
    print(f"✗ Error: {e}")

# Check for rows with NULL or unusual event_type values
unusual_sql = """
SELECT 
    COUNT(*) as total_rows,
    SUM(CASE WHEN event_type IS NULL THEN 1 ELSE 0 END) as null_event_type,
    SUM(CASE WHEN event_type LIKE 'ACS%' THEN 1 ELSE 0 END) as acs_like,
    SUM(CASE WHEN event_type LIKE '%REVASCUL%' THEN 1 ELSE 0 END) as revasc_like,
    SUM(CASE WHEN event_type LIKE '%PCI%' THEN 1 ELSE 0 END) as pci_like,
    SUM(CASE WHEN event_type LIKE '%CABG%' THEN 1 ELSE 0 END) as cabg_like
FROM tmp_ed_event_log
"""

print("\nSearching for ACS/Revascularization in event log (pattern matching):")
try:
    df_unusual = pd.read_sql(unusual_sql, conn)
    print(df_unusual.to_string(index=False))
except Exception as e:
    print(f"✗ Error: {e}")

conn.close()


2026-02-05 17:42:45,489 - src.db - INFO - Connected to database: mimiciv @ localhost



DIAGNOSTIC: Check Event Log Contents

Event types in event log:
    event_type  event_count
     ICU_ADMIT        32650
     RRT_START        18114
    VENT_START        11048
 PRESSOR_START         8321
           ACS         6195
         DEATH         4587
           PCI          962
CARDIAC_ARREST          495
          CABG          335

Searching for ACS/Revascularization in event log (pattern matching):
 total_rows  null_event_type  acs_like  revasc_like  pci_like  cabg_like
      82707                0      6195            0       962        335


C:\Users\Lab\AppData\Local\Temp\ipykernel_14212\2687462925.py:20: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_events = pd.read_sql(event_types_sql, conn)
C:\Users\Lab\AppData\Local\Temp\ipykernel_14212\2687462925.py:39: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_unusual = pd.read_sql(unusual_sql, conn)


In [11]:
from src.main import run_all

print("="*80)
print("STARTING DETERIORATION PIPELINE")
print(f"Started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("="*80)
print()

start_time = time.time()

try:
    # Run the full pipeline
    run_all("config/config.yaml")
    
    elapsed = time.time() - start_time
    print()
    print("="*80)
    print("✓ PIPELINE COMPLETED SUCCESSFULLY")
    print(f"Total time: {elapsed/60:.1f} minutes")
    print("="*80)
    
except Exception as e:
    elapsed = time.time() - start_time
    print()
    print("="*80)
    print("✗ PIPELINE FAILED")
    print(f"Error: {e}")
    print(f"Time before failure: {elapsed/60:.1f} minutes")
    print("="*80)
    raise

2026-02-05 17:57:33,334 - src.utils - INFO - Loaded configuration from: config/config.yaml
2026-02-05 17:57:33,335 - src.utils - INFO - Logging initialized (level: DEBUG)
2026-02-05 17:57:33,336 - src.main - INFO - ================================================================================
2026-02-05 17:57:33,336 - src.main - INFO - MIMIC DETERIORATION PIPELINE
2026-02-05 17:57:33,336 - src.main - INFO - ================================================================================
2026-02-05 17:57:33,337 - src.main - INFO - Config: config/config.yaml
2026-02-05 17:57:33,337 - src.main - INFO - Log file: artifacts/logs/pipeline_20260205_175733.log
2026-02-05 17:57:33,337 - src.main - INFO - Timestamp: 20260205_175733
2026-02-05 17:57:33,338 - src.main - INFO - 
2026-02-05 17:57:33,338 - src.main - INFO - Testing database connection...
2026-02-05 17:57:33,484 - src.db - INFO - Connected to database: mimiciv @ localhost
2026-02-05 17:57:33,488 - src.db - INFO - PostgreSQL version:

STARTING DETERIORATION PIPELINE
Started at: 2026-02-05 17:57:33



2026-02-05 17:57:34,569 - src.db - DEBUG - SQL executed successfully
2026-02-05 17:57:34,569 - src.db - INFO - Creating base ED cohort table - COMPLETED
2026-02-05 17:57:34,648 - src.build_base - INFO - Base cohort created: 424,952 ED visits
2026-02-05 17:57:34,649 - src.build_base - INFO - Table: tmp_base_ed_cohort
2026-02-05 17:57:34,650 - src.main - INFO - ⏱️  Step 1 completed in 1.1s

2026-02-05 17:57:34,650 - src.build_base - INFO - Validating base cohort...
2026-02-05 17:57:34,920 - src.db - DEBUG - Fetched 1 rows
2026-02-05 17:57:34,921 - src.build_base - INFO - Base Cohort Validation Results:
2026-02-05 17:57:34,922 - src.build_base - INFO -   Total ED visits: 424,952
2026-02-05 17:57:34,922 - src.build_base - INFO -   Unique patients: 205,427
2026-02-05 17:57:34,922 - src.build_base - INFO -   Unique admissions: 202,415
2026-02-05 17:57:34,923 - src.build_base - INFO -   Age: 52.9 years (range: 18-103)
2026-02-05 17:57:34,923 - src.build_base - INFO -   % Male: 45.9%
2026-02-0

2026-02-05 17:57:33,334 - src.utils - INFO - Loaded configuration from: config/config.yaml
2026-02-05 17:57:33,335 - src.utils - INFO - Logging initialized (level: DEBUG)
2026-02-05 17:57:33,336 - src.main - INFO - ================================================================================
2026-02-05 17:57:33,336 - src.main - INFO - MIMIC DETERIORATION PIPELINE
2026-02-05 17:57:33,336 - src.main - INFO - ================================================================================
2026-02-05 17:57:33,337 - src.main - INFO - Config: config/config.yaml
2026-02-05 17:57:33,337 - src.main - INFO - Log file: artifacts/logs/pipeline_20260205_175733.log
2026-02-05 17:57:33,337 - src.main - INFO - Timestamp: 20260205_175733
2026-02-05 17:57:33,338 - src.main - INFO - 
2026-02-05 17:57:33,338 - src.main - INFO - Testing database connection...
2026-02-05 17:57:33,484 - src.db - INFO - Connected to database: mimiciv @ localhost
2026-02-05 17:57:33,488 - src.db - INFO - PostgreSQL version:

STARTING DETERIORATION PIPELINE
Started at: 2026-02-05 17:57:33



2026-02-05 17:57:34,569 - src.db - DEBUG - SQL executed successfully
2026-02-05 17:57:34,569 - src.db - INFO - Creating base ED cohort table - COMPLETED
2026-02-05 17:57:34,648 - src.build_base - INFO - Base cohort created: 424,952 ED visits
2026-02-05 17:57:34,649 - src.build_base - INFO - Table: tmp_base_ed_cohort
2026-02-05 17:57:34,650 - src.main - INFO - ⏱️  Step 1 completed in 1.1s

2026-02-05 17:57:34,650 - src.build_base - INFO - Validating base cohort...
2026-02-05 17:57:34,920 - src.db - DEBUG - Fetched 1 rows
2026-02-05 17:57:34,921 - src.build_base - INFO - Base Cohort Validation Results:
2026-02-05 17:57:34,922 - src.build_base - INFO -   Total ED visits: 424,952
2026-02-05 17:57:34,922 - src.build_base - INFO -   Unique patients: 205,427
2026-02-05 17:57:34,922 - src.build_base - INFO -   Unique admissions: 202,415
2026-02-05 17:57:34,923 - src.build_base - INFO -   Age: 52.9 years (range: 18-103)
2026-02-05 17:57:34,923 - src.build_base - INFO -   % Male: 45.9%
2026-02-0


✓ PIPELINE COMPLETED SUCCESSFULLY
Total time: 11.9 minutes


✓ PIPELINE COMPLETED SUCCESSFULLY
Total time: 49.2 minutes

## Step 3.3: Verify Pipeline Outputs

Check that all tables were created successfully:

In [15]:
conn = get_conn(cfg)

expected_tables = [
    'tmp_base_ed_cohort',
    'tmp_ed_event_log',
    'tmp_ed_outcomes',
    'tmp_features_w1',
    'tmp_features_w6',
    'tmp_features_w24',
    'tmp_ecg_features_w1',
    'tmp_ecg_features_w6'
]

print("Verifying pipeline outputs...\n")
print(f"{'Table':<30s} {'Row Count':>15s}")
print("-" * 50)

cur = conn.cursor()
all_ok = True

for table in expected_tables:
    try:
        cur.execute(f"SELECT COUNT(*) FROM {table}")
        count = cur.fetchone()[0]
        status = "✓"
        print(f"{status} {table:<28s} {count:>15,}")
    except Exception as e:
        status = "✗"
        print(f"{status} {table:<28s} {'NOT FOUND':>15s}")
        all_ok = False

cur.close()
conn.close()

print()
if all_ok:
    print("✓ All tables created successfully!")
    print("\n🎉 Pipeline execution complete! Ready for dataset generation.")
else:
    print("⚠️  Some tables are missing. Check logs for errors.")

2026-02-05 18:16:44,820 - src.db - INFO - Connected to database: mimiciv @ localhost


Verifying pipeline outputs...

Table                                Row Count
--------------------------------------------------
✓ tmp_base_ed_cohort                   424,952
✓ tmp_ed_event_log                      82,707
✓ tmp_ed_outcomes                      424,952
✓ tmp_features_w1                      424,952
✓ tmp_features_w6                      424,952
✓ tmp_features_w24                     424,952
✓ tmp_ecg_features_w1                  424,952
✓ tmp_ecg_features_w6                  424,952

✓ All tables created successfully!

🎉 Pipeline execution complete! Ready for dataset generation.


## Step 3.4: Quick Data Summary

View key statistics from the pipeline outputs:

In [19]:
import pandas as pd

conn = get_conn(cfg)

# Cohort summary
print("COHORT SUMMARY")
print("=" * 70)

query = """
SELECT 
    COUNT(*) as total_visits,
    SUM(CASE WHEN was_admitted = 1 THEN 1 ELSE 0 END) as admitted,
    SUM(CASE WHEN was_admitted = 0 THEN 1 ELSE 0 END) as not_admitted,
    ROUND(AVG(age_at_ed)::NUMERIC, 1) as mean_age,
    SUM(CASE WHEN gender = 'M' THEN 1 ELSE 0 END) as male,
    SUM(CASE WHEN gender = 'F' THEN 1 ELSE 0 END) as female
FROM tmp_base_ed_cohort
"""

df = pd.read_sql(query, conn)
print(df.to_string(index=False))

# Outcome summary
print("\n\nOUTCOME SUMMARY")
print("=" * 70)

query = """
SELECT 
    COUNT(*) as total,
    SUM(deterioration_24h) as det_24h_count,
    ROUND(CAST(100.0 * AVG(deterioration_24h::float) AS NUMERIC), 2) as det_24h_pct,
    SUM(icu_24h) as icu_24h_count,
    ROUND(CAST(100.0 * AVG(icu_24h::float) AS NUMERIC), 2) as icu_24h_pct,
    SUM(death_24h) as death_24h_count,
    ROUND(CAST(100.0 * AVG(death_24h::float) AS NUMERIC), 2) as death_24h_pct,
    SUM(death_48h) as death_48h_count,
    ROUND(CAST(100.0 * AVG(death_48h::float) AS NUMERIC), 2) as death_48h_pct,
    SUM(death_hosp) as death_hosp_count,
    ROUND(CAST(100.0 * AVG(death_hosp::float) AS NUMERIC), 2) as death_hosp_pct
FROM tmp_ed_outcomes
"""

df = pd.read_sql(query, conn)
print(df.T.to_string())

conn.close()

print("\n" + "="*70)
print("✓ Pipeline verification complete")

2026-02-05 18:22:07,097 - src.db - INFO - Connected to database: mimiciv @ localhost


COHORT SUMMARY
 total_visits  admitted  not_admitted  mean_age   male  female
       424952    202990        221962      52.9 195114  229838


OUTCOME SUMMARY
                          0
total             424952.00
det_24h_count      27215.00
det_24h_pct            6.40
icu_24h_count      27022.00
icu_24h_pct            6.36
death_24h_count      595.00
death_24h_pct          0.14
death_48h_count     1119.00
death_48h_pct          0.26
death_hosp_count    4587.00
death_hosp_pct         1.08

✓ Pipeline verification complete


C:\Users\Lab\AppData\Local\Temp\ipykernel_14212\569785475.py:20: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)
C:\Users\Lab\AppData\Local\Temp\ipykernel_14212\569785475.py:43: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


---

## Alternative: Run Pipeline Steps Individually

If you prefer to run steps one at a time (useful for debugging), use the cells below instead of Step 3.2:

### Individual Steps

In [ ]:
# OPTIONAL: Run steps individually

from src.build_base import build_base_cohort
from src.build_event_log import build_event_log
from src.build_outcomes import build_outcomes
from src.build_features import build_features
from src.build_ecg_features import build_ecg_features

conn = get_conn(cfg)

try:
    # Step 1: Base cohort
    print("[1/5] Building base cohort...")
    build_base_cohort(conn, cfg)
    
    # Step 2: Event log
    print("[2/5] Extracting clinical events...")
    build_event_log(conn, cfg)
    
    # Step 3: Outcomes
    print("[3/5] Calculating outcomes...")
    build_outcomes(conn, cfg)
    
    # Step 4: Features
    print("[4/5] Extracting features...")
    build_features(conn, cfg)
    
    # Step 5: ECG features (if enabled)
    if cfg.get('ecg', {}).get('enabled', False):
        print("[5/5] Extracting ECG features...")
        build_ecg_features(conn, cfg)
    else:
        print("[5/5] ECG features skipped (not enabled)")
    
    print("\n✓ All steps completed")
    
finally:
    conn.close()

---

**✋ CHECKPOINT:** Before proceeding to Part 4, ensure:
- ✓ Pipeline completed without errors
- ✓ All 8 tmp_* tables created
- ✓ Row counts look reasonable

Continue to Part 4 for dataset generation!

---